# 05 MLflow Tracking and Registry

This notebook reviews training runs, registers existing local adapters, and compares adapter experiment metadata.

```mermaid
flowchart LR
    A[Training notebooks] --> B[MLflow experiment]
    B --> C[Run params and metrics]
    B --> D[Adapter artifacts]
    D --> E[MLflow Model Registry]
```

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "PROJECT_SPEC.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))

from llmops_demo.settings import settings, ensure_dirs

cfg = settings()
print(f"Project root: {PROJECT_ROOT}")
print(f"Base model: {cfg.base_model}")
print(f"Adapters: {cfg.adapters}")

## Configure MLflow

For local notebooks, `MLFLOW_TRACKING_URI=file:./mlruns` works without any service. With `make up`, use `http://localhost:5000`.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(cfg.mlflow_tracking_uri)
mlflow.set_experiment(cfg.mlflow_experiment_name)
client = MlflowClient()
experiment = client.get_experiment_by_name(cfg.mlflow_experiment_name)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else "not created yet")

## Register local adapters

Expected output: one registered model per adapter, named with `MLFLOW_REGISTERED_MODEL_PREFIX`.

In [ ]:
from scripts.register_mlflow import register_local_adapter

for adapter in cfg.adapters:
    adapter_path = cfg.adapter_dir / adapter
    if adapter_path.exists():
        register_local_adapter(adapter, adapter_path, cfg)
    else:
        print(f"Skipping {adapter}: {adapter_path} does not exist yet")

## Compare experiment runs

The table shows recent runs, adapter tags, and common LoRA parameters.

In [ ]:
import pandas as pd

experiment = client.get_experiment_by_name(cfg.mlflow_experiment_name)
if experiment:
    runs = mlflow.search_runs([experiment.experiment_id])
    cols = [
        "run_id",
        "status",
        "tags.adapter_name",
        "params.lora_r",
        "params.learning_rate",
        "metrics.mean_keyword_score",
    ]
    display(runs[[c for c in cols if c in runs.columns]].head(20))
else:
    print("No experiment found yet.")